In [11]:
import pandas as pd

park = pd.read_csv(
    "../data/nationalparksname.csv",
    encoding="cp874"
)

park.head()

,จังหวัด,อุทยานแห่งชาติ,URL
0,กำแพงเพชร,คลองลาน,https://portal.dnp.go.th/Content/nationalpark?...
1,กำแพงเพชร,คลองวังเจ้า,https://portal.dnp.go.th/Content/nationalpark?...
2,กำแพงเพชร,แม่วงก์,https://portal.dnp.go.th/Content/nationalpark?...
3,เชียงราย,ขุนแจ,https://portal.dnp.go.th/Content/nationalpark?...
4,เชียงราย,ดอยหลวง,https://portal.dnp.go.th/Content/nationalpark?...


1. ตรวจความสัมพันธ์ระหว่างอุทยานกับจังหวัด

ตรวจความครบถ้วนของข้อมูลรายชื่ออุทยานและจังหวัด
รวมถึงตรวจว่าอุทยานหนึ่งแห่งครอบคลุมมากกว่าหนึ่งจังหวัดหรือไม่
ก่อนนำข้อมูลไปเชื่อมกับข้อมูลปริมาณฝน

In [15]:
park.columns = park.columns.str.strip()

park["จังหวัด"] = park["จังหวัด"].str.strip()
park["อุทยานแห่งชาติ"] = park["อุทยานแห่งชาติ"].str.strip()

print("ขนาดข้อมูล:")
print(park.shape)

print("\nจำนวนจังหวัด:")
print(park["จังหวัด"].nunique())

print("\nจำนวนอุทยาน:")
print(park["อุทยานแห่งชาติ"].nunique())

print("\nMissing Values:")
print(park.isna().sum())

print("\nจำนวนข้อมูลซ้ำ:")
print(park.duplicated().sum())

province_per_park = (
    park.groupby("อุทยานแห่งชาติ")["จังหวัด"]
    .nunique()
)

print("\nจำนวนอุทยานที่อยู่มากกว่า 1 จังหวัด:")
print((province_per_park > 1).sum())

display(
    province_per_park[province_per_park > 1]
)

ขนาดข้อมูล:
(156, 3)

จำนวนจังหวัด:
50

จำนวนอุทยาน:
156

Missing Values:
จังหวัด           0
อุทยานแห่งชาติ    0
URL               0
dtype: int64

จำนวนข้อมูลซ้ำ:
0

จำนวนอุทยานที่อยู่มากกว่า 1 จังหวัด:
0


Series([], Name: จังหวัด, dtype: int64)

ในความเป็นจริงบางอุทยานครอบคลุมหลายจังหวัด แต่ Dataset นี้กำหนด จังหวัดให้อุทยานละ 1 จังหวัด น่าจะเป็นจังหวัดหลักที่ใช้ในชุดข้อมูล

2. ตรวจว่าชื่ออุทยานและจังหวัดเชื่อมกับอีก 2 Dataset ได้ไหม
ตรวจสอบว่าชื่ออุทยานสามารถเชื่อมกับข้อมูลนักท่องเที่ยว
และชื่อจังหวัดสามารถเชื่อมกับข้อมูลปริมาณฝนได้ครบถ้วนหรือไม่

In [16]:
# โหลดข้อมูล Clean ที่ทำไว้ก่อนหน้า
tourism = pd.read_csv("../data/processed/tourism_clean.csv")
weather = pd.read_csv("../data/processed/weather_clean.csv")

# ชื่ออุทยานใน Tourism แต่หาไม่เจอใน Park dataset
parks_not_matched = sorted(
    set(tourism["park_name"].unique()) -
    set(park["อุทยานแห่งชาติ"].unique())
)

print("จำนวนชื่ออุทยานที่ยังจับคู่ไม่ได้:")
print(len(parks_not_matched))

print("\nรายชื่อ:")
print(parks_not_matched)


# จังหวัดใน Park dataset แต่หาไม่เจอใน Weather
provinces_not_matched = sorted(
    set(park["จังหวัด"].unique()) -
    set(weather["province"].unique())
)

print("\nจำนวนจังหวัดที่ยังจับคู่กับ Weather ไม่ได้:")
print(len(provinces_not_matched))

print("\nรายชื่อจังหวัด:")
print(provinces_not_matched)

จำนวนชื่ออุทยานที่ยังจับคู่ไม่ได้:
44

รายชื่อ:
['ดอยสอยมาลัย', 'ดอยสอยมาลัย-ไม้กลายเป็นหิน', 'ดอยสุเทพ-ปุย', 'ดอยเวียงผา', 'ต้นสักใหญ่ (คลองตรอน)', 'ถ้ำปลา-น้ำตกผาเสื่อ', 'ถ้ำผาไท', 'ถ้ำหลวง-ขุนน้ำนางนอน', 'ธารเสด็จ-เกาะพะงัน', 'นันทบุรี', 'นายูง-น้ำโสม', 'น้ำตกซีโป', 'น้ำตกบัวตอง-น้ำพุเจ็ดสี', 'น้ำตกพาเจริญ', 'บูโด-สุไหงปาดี', 'ผาแดง (เชียงดาว)', 'ภูชี้ฟ้า', 'ภูผายา', 'ภูผาเทิบ (มุกดาหาร)', 'ภูสวนทราย (นาแห้ว)', 'ภูหินจอมธาตุ-ภูพระบาท', 'ภูเก้า-ภูพานคำ', 'ลำน้ำกก', 'สันกาลาคีรี', 'หาดขนอม-หมู่เกาะทะเลใต้', 'หาดนพรัตน์ธารา-หมู่เกาะพีพี', 'ออบขาน', 'อ่าวมะนาว-เขาตันหยง', 'อ่าวสยาม', 'เขลางค์บรรพต', 'เขาคิชฌกูฎ', 'เขาชะเมา-เขาวง', 'เขาปู่-เขาย่า', 'เขาลำปี-หาดท้ายเหมือง', 'เขาหลัก-ลำรู่', 'เขาแหลมหญ้า-หมู่เกาะเสม็ด', 'เจ็ดคด-โป่งก้อนเส้า', 'แก่งเจ็ดแคว', 'แควน้อย', 'แม่ยวมฝั่งซ้าย', 'แม่สะเรียง', 'แม่เงา', 'แม่โถ', 'ไม้กลายเป็นหิน']

จำนวนจังหวัดที่ยังจับคู่กับ Weather ไม่ได้:
0

รายชื่อจังหวัด:
[]


3. ปรับรูปแบบชื่ออุทยานก่อนการเชื่อมข้อมูล

ปรับรูปแบบชื่ออุทยานในข้อมูลนักท่องเที่ยวและข้อมูลจังหวัดให้เป็นมาตรฐานเดียวกัน
เพื่อแยกปัญหาที่เกิดจากรูปแบบการเขียนออกจากกรณีที่เป็นชื่ออุทยานคนละชื่อจริง

In [17]:
import re

def normalize_park_name(name):
    name = str(name).strip()

    # Unicode ให้เป็นรูปแบบเดียวกัน
    name = pd.Series([name]).str.normalize("NFC").iloc[0]

    # ลบคำว่า อุทยานแห่งชาติ ถ้ามีอยู่หน้าชื่อ
    name = re.sub(r"^อุทยานแห่งชาติ\s*", "", name)

    # ลบสถานะเตรียมการ
    name = re.sub(r"\s*\(เตรียมการฯ?\)", "", name)

    # ลบช่องว่างล่องหน
    name = name.replace("\u200b", "")

    # จัดรูปแบบขีดให้เหมือนกัน
    name = re.sub(r"\s*[-–—]\s*", "-", name)

    # ช่องว่างหลายตัว → เหลือหนึ่งตัว
    name = re.sub(r"\s+", " ", name)

    return name.strip()


tourism["park_key"] = tourism["park_name"].apply(normalize_park_name)
park["park_key"] = park["อุทยานแห่งชาติ"].apply(normalize_park_name)


parks_not_matched_clean = sorted(
    set(tourism["park_key"].unique()) -
    set(park["park_key"].unique())
)

print("ก่อนปรับรูปแบบ:", len(parks_not_matched))
print("หลังปรับรูปแบบ:", len(parks_not_matched_clean))

print("\nชื่อที่ยังจับคู่ไม่ได้:")
print(parks_not_matched_clean)

ก่อนปรับรูปแบบ: 44
หลังปรับรูปแบบ: 10

ชื่อที่ยังจับคู่ไม่ได้:
['ดอยสอยมาลัย', 'ต้นสักใหญ่ (คลองตรอน)', 'ผาแดง (เชียงดาว)', 'ภูผายา', 'ภูผาเทิบ (มุกดาหาร)', 'ภูสวนทราย (นาแห้ว)', 'เขาคิชฌกูฎ', 'แก่งเจ็ดแคว', 'แม่ยวมฝั่งซ้าย', 'ไม้กลายเป็นหิน']


4. ตรวจชื่ออุทยานที่ยังจับคู่ไม่ได้

ค้นหาชื่ออุทยานในชุดข้อมูลจังหวัดที่มีความใกล้เคียงกับ
ชื่ออุทยานที่ยังจับคู่ไม่ได้ เพื่อพิจารณาการเปลี่ยนชื่อหรือการสะกดที่แตกต่างกัน
ก่อนทำการเชื่อมข้อมูล

In [18]:
from difflib import SequenceMatcher

park_names = park["park_key"].dropna().unique()

results = []

for name in parks_not_matched_clean:
    
    best_match = None
    best_score = 0
    
    for candidate in park_names:
        score = SequenceMatcher(
            None,
            name,
            candidate
        ).ratio()
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    results.append({
        "tourism_name": name,
        "closest_park_name": best_match,
        "similarity": round(best_score, 2)
    })

name_matches = pd.DataFrame(results)

name_matches.sort_values(
    "similarity",
    ascending=False
)

,tourism_name,closest_park_name,similarity
6,เขาคิชฌกูฎ,เขาคิชฌกูฏ,0.90
3,ภูผายา,ภูผายล,0.83
9,ไม้กลายเป็นหิน,ดอยสอยมาลัย-ไม้กลายเป็นหิน,0.70
5,ภูสวนทราย (นาแห้ว),ภูสวนทราย,0.67
1,ต้นสักใหญ่ (คลองตรอน),ต้นสักใหญ่,0.65
0,ดอยสอยมาลัย,ดอยสอยมาลัย-ไม้กลายเป็นหิน,0.59
4,ภูผาเทิบ (มุกดาหาร),ภูผาเทิบ,0.59
8,แม่ยวมฝั่งซ้าย,แม่ยม,0.53
2,ผาแดง (เชียงดาว),ผาแดง,0.48
7,แก่งเจ็ดแคว,แก่งกระจาน,0.48


5. จัดการชื่ออุทยานที่ตรวจสอบแล้ว

กำหนดชื่อมาตรฐานเฉพาะกรณีที่ตรวจสอบได้ว่าเป็นอุทยานเดียวกัน
โดยไม่ใช้ผล Fuzzy Matching รวมชื่อโดยอัตโนมัติ

In [19]:
park_name_mapping = {
    "ภูสวนทราย (นาแห้ว)": "ภูสวนทราย",
    "ต้นสักใหญ่ (คลองตรอน)": "ต้นสักใหญ่",
    "ภูผาเทิบ (มุกดาหาร)": "ภูผาเทิบ",
    "ผาแดง (เชียงดาว)": "ผาแดง",
    "ดอยสอยมาลัย": "ดอยสอยมาลัย-ไม้กลายเป็นหิน",
    "ไม้กลายเป็นหิน": "ดอยสอยมาลัย-ไม้กลายเป็นหิน"
}

tourism["park_key_final"] = (
    tourism["park_key"]
    .replace(park_name_mapping)
)

remaining_unmatched = sorted(
    set(tourism["park_key_final"].unique()) -
    set(park["park_key"].unique())
)

print("จำนวนชื่อที่ยังจับคู่ไม่ได้:")
print(len(remaining_unmatched))

print("\nรายชื่อ:")
print(remaining_unmatched)

จำนวนชื่อที่ยังจับคู่ไม่ได้:
4

รายชื่อ:
['ภูผายา', 'เขาคิชฌกูฎ', 'แก่งเจ็ดแคว', 'แม่ยวมฝั่งซ้าย']


6. ตรวจรายละเอียดชื่ออุทยานที่ยังจับคู่ไม่ได้

ตรวจปี สังกัด และชื่อเดิมของอุทยานที่ไม่สามารถจับคู่กับ
รายชื่ออุทยานปัจจุบันได้ เพื่อระบุจังหวัดสำหรับเชื่อมข้อมูลสภาพอากาศ

In [20]:
tourism[
    tourism["park_key_final"].isin(remaining_unmatched)
][
    ["park_name", "park_key_final", "office", "fiscal_year_be"]
].drop_duplicates().sort_values(
    ["park_key_final", "fiscal_year_be"]
)

,park_name,park_key_final,office,fiscal_year_be
1829,ภูผายา,ภูผายา,สำนักบริหารพื้นที่อนุรักษ์ที่ 10,2560
5537,ภูผายา,ภูผายา,สำนักบริหารพื้นที่อนุรักษ์ที่ 10,2562
105,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2559
1874,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2560
3724,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2561
5582,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2562
7432,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2563
9292,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2,2564
11152,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2 (ศรีราชา),2565
13012,เขาคิชฌกูฎ,เขาคิชฌกูฎ,สำนักบริหารพื้นที่อนุรักษ์ที่ 2 (ศรีราชา),2566


 จัดการอุทยานที่ไม่สามารถจับคู่จังหวัดได้โดยตรง

กำหนดจังหวัดให้กับชื่ออุทยานเก่าหรือชื่อที่มีรูปแบบแตกต่างจากชุดข้อมูลปัจจุบัน
โดยอ้างอิงข้อมูลของกรมอุทยานแห่งชาติ และแยกกรณีอุทยานที่ครอบคลุมหลายจังหวัด
ไว้สำหรับจัดการเพิ่มเติม

In [40]:
manual_province = {
    "เขาคิชฌกูฎ": "จันทบุรี",
    "แก่งเจ็ดแคว": "พิษณุโลก",
    "แม่ยวมฝั่งซ้าย": "แม่ฮ่องสอน"
}

สร้างตารางจับคู่อุทยานกับจังหวัด

สร้างตารางสำหรับเชื่อมชื่ออุทยานกับจังหวัด โดยใช้ข้อมูลรายชื่ออุทยานเป็นหลัก
และเติมจังหวัดให้กรณีชื่อเก่าหรือชื่อที่ไม่สามารถจับคู่ได้โดยตรง

In [41]:
# ตารางอุทยาน -> จังหวัด จาก Dataset หลัก
park_lookup = (
    park[["park_key", "จังหวัด"]]
    .drop_duplicates()
    .rename(columns={
        "park_key": "park_name",
        "จังหวัด": "province"
    })
)

# เติม 3 อุทยานที่จับคู่ไม่ได้โดยตรง
manual_lookup = pd.DataFrame({
    "park_name": [
        "เขาคิชฌกูฎ",
        "แก่งเจ็ดแคว",
        "แม่ยวมฝั่งซ้าย"
    ],
    "province": [
        "จันทบุรี",
        "พิษณุโลก",
        "แม่ฮ่องสอน"
    ]
})

park_lookup = pd.concat(
    [park_lookup, manual_lookup],
    ignore_index=True
)

park_lookup.head()

,park_name,province
0,คลองลาน,กำแพงเพชร
1,คลองวังเจ้า,กำแพงเพชร
2,แม่วงก์,กำแพงเพชร
3,ขุนแจ,เชียงราย
4,ดอยหลวง,เชียงราย


In [31]:
phu_pha_ya_lookup = pd.DataFrame({
    "park_name": ["ภูผายา", "ภูผายา", "ภูผายา"],
    "province": ["เลย", "หนองบัวลำภู", "อุดรธานี"]
})

park_lookup = pd.concat(
    [park_lookup, phu_pha_ya_lookup],
    ignore_index=True
)

tourism_park_names = set(tourism["park_key_final"].unique())
mapped_park_names = set(park_lookup["park_name"].unique())

still_unmatched = sorted(
    tourism_park_names - mapped_park_names
)

print("จำนวนอุทยานที่ยังไม่มีจังหวัด:")
print(len(still_unmatched))

print("\nรายชื่อ:")
print(still_unmatched)

จำนวนอุทยานที่ยังไม่มีจังหวัด:
0

รายชื่อ:
[]


9. สร้างข้อมูลปริมาณฝนระดับอุทยาน

เชื่อมข้อมูลจังหวัดของแต่ละอุทยานกับข้อมูลปริมาณฝนรายเดือน
และคำนวณค่าเฉลี่ยสำหรับอุทยานที่ครอบคลุมมากกว่าหนึ่งจังหวัด
เพื่อให้ได้ข้อมูลหนึ่งแถวต่ออุทยานต่อเดือน

In [42]:
# เชื่อมอุทยานกับข้อมูลฝนตามจังหวัด
park_weather = park_lookup.merge(
    weather,
    on="province",
    how="left"
)

# กรณีอุทยานมีหลายจังหวัด เช่น ภูผายา
# เฉลี่ยค่าฝนของจังหวัดเหล่านั้นให้เหลือ 1 ค่า/อุทยาน/เดือน
park_weather_monthly = (
    park_weather
    .groupby(
        ["park_name", "year", "month", "date"],
        as_index=False
    )
    .agg(
        min_rain=("min_rain", "mean"),
        max_rain=("max_rain", "mean"),
        avg_rain=("avg_rain", "mean")
    )
)

park_weather_monthly.head()

,park_name,year,month,date,min_rain,max_rain,avg_rain
0,กุยบุรี,2018,1,2018-01-01,14.500000,54.709999,28.620681
1,กุยบุรี,2018,2,2018-02-01,44.020000,81.910004,67.612976
2,กุยบุรี,2018,3,2018-03-01,13.300000,63.369999,28.563502
3,กุยบุรี,2018,4,2018-04-01,124.010002,223.779999,174.146624
4,กุยบุรี,2018,5,2018-05-01,76.709999,260.809998,136.703389


In [43]:
#เช็คข้อมูลซ้ำ
print("จำนวนข้อมูลซ้ำ:")
print(
    park_weather_monthly.duplicated(
        subset=["park_name", "year", "month"]
    ).sum()
)

print("\nMissing Values:")
print(park_weather_monthly.isna().sum())

จำนวนข้อมูลซ้ำ:
0

Missing Values:
park_name    0
year         0
month        0
date         0
min_rain     0
max_rain     0
avg_rain     0
dtype: int64


10. เชื่อมข้อมูลนักท่องเที่ยวกับข้อมูลปริมาณฝน

เชื่อมข้อมูลนักท่องเที่ยวรายเดือนกับข้อมูลปริมาณฝนของแต่ละอุทยาน
โดยใช้ชื่ออุทยานและวันที่เป็นตัวเชื่อม เพื่อเตรียมข้อมูลสำหรับวิเคราะห์
ความสัมพันธ์ระหว่างสภาพฝนและจำนวนผู้เข้าชม

In [44]:
# ทำให้ date ของทั้งสอง Dataset เป็นชนิด datetime เหมือนกัน
tourism["date"] = pd.to_datetime(tourism["date"])
park_weather_monthly["date"] = pd.to_datetime(
    park_weather_monthly["date"]
)

# เชื่อม Tourism กับ Weather
tourism_weather = tourism.merge(
    park_weather_monthly[
        [
            "park_name",
            "date",
            "min_rain",
            "max_rain",
            "avg_rain"
        ]
    ],
    left_on=["park_key_final", "date"],
    right_on=["park_name", "date"],
    how="inner"
)

tourism_weather.head()

,fiscal_year_be,park_name_x,office,month_th,visitors,month_num,calendar_year_be,calendar_year_ce,date,park_key,park_key_final,park_name_y,min_rain,max_rain,avg_rain
0,2561,กุยบุรี,สำนักบริหารพื้นที่อนุรักษ์ที่ 3 สาขาเพชรบุรี,ม.ค.,2199,1,2561,2018,2018-01-01,กุยบุรี,กุยบุรี,กุยบุรี,14.5,54.709999,28.620681
1,2561,ขุนขาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 16,ม.ค.,16242,1,2561,2018,2018-01-01,ขุนขาน,ขุนขาน,ขุนขาน,0.3,30.400000,6.695410
2,2561,ขุนน่าน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,523,1,2561,2018,2018-01-01,ขุนน่าน,ขุนน่าน,ขุนน่าน,11.4,30.780001,17.231855
3,2561,ขุนพะวอ,สำนักบริหารพื้นที่อนุรักษ์ที่ 14,ม.ค.,521,1,2561,2018,2018-01-01,ขุนพะวอ,ขุนพะวอ,ขุนพะวอ,0.0,32.000000,9.917311
4,2561,ขุนสถาน,สำนักบริหารพื้นที่อนุรักษ์ที่ 13,ม.ค.,9956,1,2561,2018,2018-01-01,ขุนสถาน,ขุนสถาน,ขุนสถาน,11.4,30.780001,17.231855


In [45]:
print("ขนาดข้อมูลหลัง Merge:")
print(tourism_weather.shape)

print("\nช่วงวันที่:")
print(
    tourism_weather["date"].min(),
    "ถึง",
    tourism_weather["date"].max()
)

print("\nMissing Values:")
print(
    tourism_weather[
        ["visitors", "min_rain", "max_rain", "avg_rain"]
    ].isna().sum()
)

print("\nจำนวนข้อมูลซ้ำ:")
print(
    tourism_weather.duplicated(
        subset=["park_key_final", "date"]
    ).sum()
)

ขนาดข้อมูลหลัง Merge:
(14427, 15)

ช่วงวันที่:
2018-01-01 00:00:00 ถึง 2025-09-01 00:00:00

Missing Values:
visitors    0
min_rain    0
max_rain    0
avg_rain    0
dtype: int64

จำนวนข้อมูลซ้ำ:
0


11. บันทึก Integrated Dataset

บันทึกข้อมูลนักท่องเที่ยวและปริมาณฝนที่ผ่านการเชื่อมและตรวจสอบแล้ว
เพื่อใช้เป็น Dataset หลักสำหรับการวิเคราะห์ในขั้นตอนถัดไป

In [47]:
output_path = "../data/processed/tourism_weather_clean.csv"

tourism_weather.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("บันทึกเรียบร้อย:", output_path)
print("ขนาดข้อมูล:", tourism_weather.shape)

บันทึกเรียบร้อย: ../data/processed/tourism_weather_clean.csv
ขนาดข้อมูล: (14427, 15)
